На 0.png есть только положение 0, 5, 4 метки
Предпалогаю что плитка - 3 аруко метки - 0.52 см
Правый ближний - id 0,
правая дальняя - 4,


ближняя - 5

id    coordination    смещение
0
4
5


In [11]:
import cv2
import numpy as np

# Параметры камеры из калибровки
fx, fy = 1380.76188376, 1378.32004896
cx, cy = 985.37631507, 557.57176819
camera_matrix = np.array([[fx, 0, cx],
                          [0, fy, cy],
                          [0, 0, 1]], dtype=np.float32)

# Коэффициенты дисторсии
dist_coeffs = np.array([-0.00573079, -0.05853685, 0.00310127, 0.00000221, 0.0], dtype=np.float32)

# Размер метки ArUco (в метрах)
marker_size = 0.17  # 17 см, изменить при необходимости
# Длина плитки
len_brick = 0.5 # в метрах


# Координаты ArUco меток относительно центра поля (в метрах)
# Пример: {id: [x, y, z]}
marker_positions = {
    0: [-3 * len_brick - 0.075, 0 - 0.085, 0.0],   # Метка с ID 0
    4: [-3 * len_brick - 0.145, 4 * len_brick + 18.5, 0.0],  # Метка с ID 4
    5: [3 * len_brick + 0.055, 4 * len_brick + 0.125, 0.0],  # Метка с ID 5
}

# Функция для вычисления положения камеры
def estimate_camera_pose(corners, ids, camera_matrix, dist_coeffs, marker_positions, marker_size):
    if ids is None or len(ids) == 0:
        print("Метки не обнаружены")
        return None

    # Определение 3D точек углов метки
    obj_points = np.array([[-marker_size / 2, marker_size / 2, 0],
                           [marker_size / 2, marker_size / 2, 0],
                           [marker_size / 2, -marker_size / 2, 0],
                           [-marker_size / 2, -marker_size / 2, 0]], dtype=np.float32)

    camera_positions = []

    # Для каждой обнаруженной метки
    for i, marker_id in enumerate(ids.flatten()):
        if marker_id not in marker_positions:
            continue

        # Оценка позы метки
        corners_marker = corners[i].reshape((4, 2))
        success, rvec, tvec = cv2.solvePnP(obj_points, corners_marker, camera_matrix, dist_coeffs)

        if not success:
            continue

        # Преобразование вектора вращения в матрицу вращения
        rmat, _ = cv2.Rodrigues(rvec)

        # Положение метки в системе координат поля
        marker_pos = np.array(marker_positions[marker_id], dtype=np.float32)

        # Вычисление положения камеры в системе координат метки
        camera_pos_in_marker = -np.dot(rmat.T, tvec).flatten()

        # Положение камеры в системе координат поля
        camera_pos_in_world = camera_pos_in_marker + marker_pos

        # Ориентация камеры (матрица вращения)
        camera_rotation = rmat.T

        camera_positions.append((marker_id, camera_pos_in_world, camera_rotation))

    if not camera_positions:
        print("Не удалось вычислить положение камеры")
        return None

    # Усреднение положений камеры (если обнаружено несколько меток)
    avg_camera_pos = np.mean([pos for _, pos, _ in camera_positions], axis=0)

    # Для простоты берем ориентацию от первой метки
    avg_camera_rot = camera_positions[0][2]

    return avg_camera_pos, avg_camera_rot

def main():
    # Загрузка словаря ArUco
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_100)
    parameters = cv2.aruco.DetectorParameters()
    detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

    # Захват видео с камеры
    frame = cv2.imread('../../datasets/test/cam0/0.png')

    # Обнаружение меток
    corners, ids, _ = detector.detectMarkers(frame)

    # Отрисовка меток на кадре
    if ids is not None:
        cv2.aruco.drawDetectedMarkers(frame, corners, ids)

    # Вычисление положения камеры
    result = estimate_camera_pose(corners, ids, camera_matrix, dist_coeffs, marker_positions, marker_size)

    if result is not None:
        camera_pos, camera_rot = result
        print(f"Положение камеры: x={camera_pos[0]:.2f}, y={camera_pos[1]:.2f}, z={camera_pos[2]:.2f} м")
        print("Матрица вращения камеры:")
        print(camera_rot)

    # Отображение кадра
    cv2.imshow('Frame', frame)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Положение камеры: x=-1.16, y=3.91, z=2.01 м
Матрица вращения камеры:
[[-0.99946272  0.02818325 -0.01673243]
 [ 0.00259306  0.57689944  0.81681106]
 [ 0.03267332  0.81632882 -0.57666257]]
